In [25]:
# SummarizationMiddleware 用来压缩 agent 的对话历史，帮助对话较长时控制上下文长度。
# 它会监测消息数量或 token 数。达到 trigger 条件后，中间件把较早的消息交给指定的 model 概括，再用摘要替代那部分历史；keep 则指定保留多少近期消息。

from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from rich import print as rprint
from langchain_core.tools import tool 
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

model = ChatOpenAI(
    model="MiniMax-M3",
    api_key='sk-cp-GsTRU_GlFRX_WPY8cf8IqBTDtrVJP4zl4EWLZ8D7627lrDtV-MbWYnWWK7rmvT7ZRluuMtUL7VP2sCD2uq6SzkpIlY0EAJtCIHW2uSGAT4QKYxX9mJQmK8I',
    base_url="https://api.minimaxi.com/v1",
)

agent =create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=[("tokens", 100),("messages",5)],
            keep=("messages", 1),
            summary_prompt="帮我总结一下之前的对话内容，要求简明扼要，保留关键信息。消息列表如下：\n{messages}\n"
        )
    ]
)
messages = [
    SystemMessage(content="你是一个非常友好的AI助手"),
    HumanMessage(content="你好啊，我是老王，你是谁？"),
    AIMessage(content="你好，老王！我是一小王"),
    HumanMessage(content="好的小王，很高兴认识你。"),
    AIMessage(content="你高兴的太早了"),
    HumanMessage(content="呵呵，你什么意思？"),
]

res = agent.invoke({
    "messages": messages
})

for msg in res["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

<think>用户要求总结之前的的对话内容。我需要简明扼要，保留关键信息。

对话内容：
- System: AI助手介绍
- Human: 老王自我介绍，询问AI身份
- AI: 自称"小王"
- Human: 礼貌回应
- AI: 略带调侃的回复

关键信息：人物-老王和小王，初次见面，简单寒暄。</think>

**对话概要：**

老王与AI"小王"初次相识。老王自我介绍并询问AI身份，AI回复自称"小王"。老王礼貌问好，AI则以"你高兴得太早了"回应，带有些许调侃意味。整体为简短的初次见面寒暄。
================================ Human Message =================================

呵呵，你什么意思？
================================== Ai Message ==================================

<think>The user is asking "呵呵，你什么意思？" (Hehe, what do you mean?) in response to the summary of the conversation. They're reacting to my previous response which said "你高兴得太早了" (you're celebrating too early) - that was apparently a teasing/mischievous response to their polite greeting.

They're asking what I meant by that teasing comment. I should respond in a lighthearted, playful way that matches the casual tone of the conversation, whi